In [ ]:
# MCP SCENARIO: “Smart Banking Support Assistant”
# 🧩 Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
# 👉 Instead of calling customer care, customers use an AI Banking Support Bot.

# 🤖 What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 
# |  |  | 



# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI


# ============================================
# LOAD ENV
# ============================================

load_dotenv()

client = OpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1"
)


# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: LLM ANALYSIS
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are a banking customer support assistant.

Analyze the customer problem and respond ONLY in JSON format:

{{
 "create_ticket": true/false,
 "category": "Card Services/Online Banking/Loans/Transactions/General",
 "priority": "High/Medium",
 "guidance": "short helpful solution"
}}

Customer Query:
"{user_input}"
"""

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[
            {"role": "user", "content": prompt}
        ],

        temperature=0
    )

    output = response.choices[0].message.content


    try:
        parsed = json.loads(output)

    except:

        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium",
            "guidance": "Customer support will assist shortly."
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Customer Query:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)


    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)


        return f"""

✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}

Category: {result['category']}
Priority: {result['priority']}

📌 Guidance:
{decision['guidance']}

Our banking specialist will contact you shortly.
"""


    else:

        return f"""

🤖 Instant Help:

{decision['guidance']}

No escalation required.
"""


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🏦 FinTrust Banking Support Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask Banking Bot: ")

    if user_input.lower() == "exit":

        print("👋 Session ended")
        break


    response = mcp_agent(user_input)

    print(response)

🏦 FinTrust Banking Support Assistant Started (type 'exit')


🧠 Customer Query: My credit card is not working
🤖 LLM Decision: {'create_ticket': True, 'category': 'Card Services', 'priority': 'High', 'guidance': "Please try to verify your credit card details, ensure your card is activated and not expired. If issue persists, we'll assist you in resolving the issue."}
📦 MCP Payload: {'issue': 'My credit card is not working', 'priority': 'High', 'category': 'Card Services'}


✅ Support Ticket Created!

Ticket ID: BNK1000

Category: Card Services
Priority: High

📌 Guidance:
Please try to verify your credit card details, ensure your card is activated and not expired. If issue persists, we'll assist you in resolving the issue.

Our banking specialist will contact you shortly.

👋 Session ended
